# CD-HIT 蛋白序列关联样本元数据

在项目根目录运行本 Notebook。以 CD-HIT 的代表 FASTA 为输入，每条蛋白输出一行。
如果当前内核缺少 openpyxl，在代码单元中运行 %pip install openpyxl 后再执行本流程。

- **fasta_sample_id** 来自蛋白 ID，去掉末尾的七位基因编号。
- **s1_sample_id** 和 **s1_accession** 分别是补充表 S1 的样本编号和数据登录号。
- **s1_row** 是样本在原工作表中的行号，便于回查。
- 温度字段保留原表中的 N/A。

In [1]:
import csv
from itertools import zip_longest
import os
from pathlib import Path
import re
from tempfile import NamedTemporaryFile

from openpyxl import load_workbook

project = Path.cwd()
fasta_path = project / 'output/cdhit/meer1000M_cd90.pep.fa'
s1_path = project / 'dataset/MEER/DSGC_Table_S1.xlsx'
alias_path = project / 'dataset/论文附录数据/07_DSGC_CHM2026_TableS1_metagenome_metadata_latlon.xlsx'
output_path = fasta_path.with_name('meer1000M_cd90_DSGC_Table_S1.csv')

## 1. 建立样本对应关系

优先匹配原样本 ID，其次统一下划线与连字符并检查对应关系唯一。
FDZ064-BuG10-12-7 根据两版 S1 第 420 行对应到 FDZ064-BuG10-12。
两行的经纬度、深度、材料和总测序量一致。导出的登录号和元数据取自 DSGC_Table_S1。

In [2]:
def read_s1(path):
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        rows = workbook['S1'].iter_rows(min_row=4, values_only=True)
        return {n: row for n, row in enumerate(rows, start=4) if row[1]}
    finally:
        workbook.close()


def sample_fields(cell):
    stem, separator, accession = cell.strip().rpartition(' (')
    if separator and accession.endswith(')'):
        return stem.strip(), accession[:-1]
    return cell.strip(), ''


def normalize_sample_id(sample_id):
    return re.sub(r'[-_]+', '-', sample_id).strip('-')


s1_rows = read_s1(s1_path)
samples = {}
normalized_samples = {}
for row_number, row in s1_rows.items():
    sample_id, accession = sample_fields(row[1])
    normalized = normalize_sample_id(sample_id)
    if sample_id in samples or normalized in normalized_samples:
        raise ValueError(f'S1 样本编号不唯一：{sample_id}')
    samples[sample_id] = (row_number, accession, row)
    normalized_samples[normalized] = sample_id

original_row = s1_rows[420]
alias_row = read_s1(alias_path)[420]
original_id, _ = sample_fields(original_row[1])
alias_id, _ = sample_fields(alias_row[1])
if (original_id, alias_id) != ('FDZ064-BuG10-12', 'FDZ064-BuG10-12-7'):
    raise ValueError('两版 S1 第 420 行的样本编号有变动')
# 两版 S1 使用不同登录号。核对这两列之外的样本元数据。
for i, (original, revised) in enumerate(zip(original_row, alias_row)):
    if i not in (1, 9) and original != revised:
        raise ValueError('两版 S1 第 420 行的样本元数据不一致')


def lookup_sample(sample_id):
    if sample_id in samples:
        return sample_id
    canonical = normalized_samples.get(normalize_sample_id(sample_id))
    if canonical is not None:
        return canonical
    if sample_id == alias_id:
        return original_id
    raise ValueError(f'找不到样本元数据：{sample_id}')

## 2. 拼接代表 FASTA 与元数据

按 FASTA 原顺序输出，每条氨基酸序列拼成一列。

In [3]:
columns = [
    'protein_id', 'fasta_sample_id', 's1_sample_id', 's1_accession',
    's1_row', 'aa_length', 'sequence', 'fasta_description',
    'ecosystem_group', 'ncbi_taxonomy',
    'isolation_source_environmental_context', 'sample_materials',
    'latitude', 'longitude', 'depth_m', 'temperature_c',
    'related_srr_ids', 'total_bases_gb', 'source_doi_and_reference',
]


def fasta_records(path):
    with path.open(encoding='utf-8') as handle:
        header = None
        parts = []
        for raw_line in handle:
            line = raw_line.strip()
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(parts)
                header = line[1:]
                parts = []
            elif line:
                parts.append(line)
        if header is not None:
            yield header, ''.join(parts)


def joined_rows():
    for header, sequence in fasta_records(fasta_path):
        protein_id, description = header.split(maxsplit=1)
        sample_id, _, gene_number = protein_id.rpartition('_')
        if not sample_id or not re.fullmatch(r'[0-9]{7}', gene_number):
            raise ValueError(f'蛋白 ID 格式异常：{protein_id}')
        if not sequence:
            raise ValueError(f'序列为空：{protein_id}')

        canonical = lookup_sample(sample_id)
        row_number, accession, metadata = samples[canonical]
        yield (
            protein_id, sample_id, canonical, accession, row_number,
            len(sequence), sequence, description, metadata[0], *metadata[2:],
        )

## 3. 导出并核验 CSV

逐行核对拼接结果。重复运行时直接核验已有 CSV。
更换输入数据后，可以调整 output_path 导出到新的文件。

In [4]:
def verify_csv(path):
    protein_ids = set()
    sample_ids = set()
    with path.open(encoding='utf-8-sig', newline='') as handle:
        reader = csv.reader(handle, strict=True)
        if next(reader, None) != columns:
            raise ValueError(f'CSV 列结构不同，请检查输出路径：{path.name}')

        for line_number, (saved, expected) in enumerate(
            zip_longest(reader, joined_rows()), start=2
        ):
            if saved is None or expected is None:
                raise ValueError(f'CSV 行数与 FASTA 不一致：第 {line_number} 行')
            expected_text = [str(value) if value is not None else '' for value in expected]
            if saved != expected_text:
                raise ValueError(f'CSV 内容与输入不一致：第 {line_number} 行')
            protein_id, _, canonical_id = saved[:3]
            if protein_id in protein_ids:
                raise ValueError(f'蛋白 ID 重复：{protein_id}')
            protein_ids.add(protein_id)
            sample_ids.add(canonical_id)

    if not protein_ids:
        raise ValueError('输入 FASTA 没有序列')
    return len(protein_ids), len(sample_ids)


def export_csv(path):
    if path.exists():
        return verify_csv(path)

    temporary_path = None
    try:
        with NamedTemporaryFile(
            mode='w', dir=path.parent, prefix=f'.{path.stem}_', suffix='.csv',
            encoding='utf-8-sig', newline='', delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            writer = csv.writer(handle)
            writer.writerow(columns)
            writer.writerows(joined_rows())

        counts = verify_csv(temporary_path)
        os.link(temporary_path, path)
        return counts
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


protein_count, sample_count = export_csv(output_path)
print(f'{output_path.name} | {protein_count:,} 条蛋白序列 | {sample_count:,} 个样本')

meer1000M_cd90_DSGC_Table_S1.csv | 386,829 条蛋白序列 | 1,121 个样本
